# DP-04 — Engineer Features for the Validation Engine

**Fase:** Data Preparation  
**Proyecto:** Healthcare AI Billing Auditor  
**Metodología:** ASUM-DM  

Creación de features derivadas a partir del Master Dataset oficial para el Motor de Reglas y el modelo CNN 1D.  
No se entrena ningún modelo ni se eliminan columnas originales.

---
## 1. Carga del Master Dataset Oficial

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)

MASTER_PATH = Path('../data/master/master_dataset.parquet')

if not MASTER_PATH.exists():
    MASTER_PATH = Path('../data/master/master_dataset.csv')
    df = pd.read_csv(MASTER_PATH)
else:
    df = pd.read_parquet(MASTER_PATH, engine='fastparquet')

original_cols = df.columns.tolist()
memory_mb = df.memory_usage(deep=True).sum() / 1024 / 1024

print(f'Master Dataset cargado: {MASTER_PATH}')
print(f'Dimensiones: {df.shape[0]:,} registros x {df.shape[1]} columnas')
print(f'Memoria: {memory_mb:.2f} MB')
print(f'\nColumnas originales: {len(original_cols)}')

Master Dataset cargado: ..\data\master\master_dataset.parquet
Dimensiones: 3,126 registros x 35 columnas
Memoria: 6.26 MB

Columnas originales: 35


---
## 2. Features Clínicas

Features derivadas de la información clínica que apoyan directamente al Motor de Reglas y al modelo.

In [7]:
# --- Feature: cups_match ---
# BR-01: ¿El código CUPS registrado coincide con el facturado?
df['cups_match'] = (
    df['codigo_cups'].fillna('') == df['codigo_cups_facturado'].fillna('')
).astype(int)
# Donde ambos son NaN, no hay comparación posible → marcar como 0
df.loc[df['codigo_cups'].isna() & df['codigo_cups_facturado'].isna(), 'cups_match'] = 0
print(f'cups_match — Distribución:\n{df["cups_match"].value_counts()}')

# --- Feature: tiene_soporte_clinico ---
# BR-02: ¿El registro tiene soporte clínico?
df['tiene_soporte_clinico'] = (df['soporte_clinico'] == 'SI').astype(int)
print(f'\ntiene_soporte_clinico — Distribución:\n{df["tiene_soporte_clinico"].value_counts()}')

# --- Feature: procedimiento_facturado ---
# ¿Existe una prefactura asociada? (id_prefactura no nulo)
df['procedimiento_facturado'] = df['id_prefactura'].notna().astype(int)
print(f'\nprocedimiento_facturado — Distribución:\n{df["procedimiento_facturado"].value_counts()}')

# --- Feature: procedimiento_registrado ---
# ¿Existe un registro en HC? (id_detalle_hc no nulo)
df['procedimiento_registrado'] = df['id_detalle_hc'].notna().astype(int)
print(f'\nprocedimiento_registrado — Distribución:\n{df["procedimiento_registrado"].value_counts()}')

# --- Feature: diagnostico_disponible ---
# ¿Tiene diagnóstico CIE-10 registrado?
df['diagnostico_disponible'] = df['diagnostico_principal_cie10'].notna().astype(int)
print(f'\ndiagnostico_disponible — Distribución:\n{df["diagnostico_disponible"].value_counts()}')

cups_match — Distribución:
cups_match
1    2784
0     342
Name: count, dtype: int64

tiene_soporte_clinico — Distribución:
tiene_soporte_clinico
1    3056
0      70
Name: count, dtype: int64

procedimiento_facturado — Distribución:
procedimiento_facturado
1    2974
0     152
Name: count, dtype: int64

procedimiento_registrado — Distribución:
procedimiento_registrado
1    3056
0      70
Name: count, dtype: int64

diagnostico_disponible — Distribución:
diagnostico_disponible
1    3126
Name: count, dtype: int64


In [8]:
# --- Features de longitud de texto ---
# Útiles para detectar descripciones vacías o muy cortas

df['len_descripcion_diagnostico'] = df['descripcion_diagnostico'].fillna('').str.len()
df['len_descripcion_hc'] = df['descripcion'].fillna('').str.len()
df['len_descripcion_servicio'] = df['descripcion_servicio_facturado'].fillna('').str.len()

print('Longitudes de texto generadas:')
print(df[['len_descripcion_diagnostico', 'len_descripcion_hc', 'len_descripcion_servicio']].describe())

Longitudes de texto generadas:
       len_descripcion_diagnostico  len_descripcion_hc  \
count                  3126.000000         3126.000000   
mean                     33.916187           25.867242   
std                      10.877088           10.887790   
min                      18.000000            0.000000   
25%                      23.000000           16.000000   
50%                      34.000000           20.000000   
75%                      43.000000           39.000000   
max                      55.000000           41.000000   

       len_descripcion_servicio  
count               3126.000000  
mean                  25.175304  
std                   11.552204  
min                    0.000000  
25%                   16.000000  
50%                   20.000000  
75%                   36.000000  
max                   41.000000  


---
## 3. Features de Facturación

Features derivadas del proceso administrativo y de facturación.

In [9]:
# --- Feature: diferencia_cantidad ---
# BR-06: Diferencia entre cantidad realizada y facturada
df['diferencia_cantidad'] = (
    df['cantidad_realizada'].fillna(0) - df['cantidad_facturada'].fillna(0)
).astype(int)
print(f'diferencia_cantidad — Distribución:\n{df["diferencia_cantidad"].value_counts()}')

# --- Feature: cantidad_coincide ---
# BR-06: ¿Coinciden las cantidades?
df['cantidad_coincide'] = (df['diferencia_cantidad'] == 0).astype(int)
# Si ambas son NaN → no aplica → 0
df.loc[df['cantidad_realizada'].isna() & df['cantidad_facturada'].isna(), 'cantidad_coincide'] = 0
print(f'\ncantidad_coincide — Distribución:\n{df["cantidad_coincide"].value_counts()}')

# --- Feature: valor_unitario_disponible ---
df['valor_unitario_disponible'] = df['valor_unitario'].notna().astype(int)

# --- Feature: valor_total_disponible ---
df['valor_total_disponible'] = df['valor_total'].notna().astype(int)

# --- Feature: servicio_facturado (alias legible) ---
df['servicio_facturado'] = df['procedimiento_facturado']  # ya creada

# --- Feature: procedimiento_no_facturado ---
# Tiene registro clínico pero NO prefactura
df['procedimiento_no_facturado'] = (
    (df['procedimiento_registrado'] == 1) & (df['procedimiento_facturado'] == 0)
).astype(int)
print(f'\nprocedimiento_no_facturado — Distribución:\n{df["procedimiento_no_facturado"].value_counts()}')

diferencia_cantidad — Distribución:
diferencia_cantidad
 0    2830
 1     131
-1     106
-2      38
 2      21
Name: count, dtype: int64

cantidad_coincide — Distribución:
cantidad_coincide
1    2830
0     296
Name: count, dtype: int64

procedimiento_no_facturado — Distribución:
procedimiento_no_facturado
0    2974
1     152
Name: count, dtype: int64


---
## 4. Features Temporales

In [10]:
# Convertir columnas de fecha a datetime
date_cols = ['fecha_atencion', 'fecha_registro', 'fecha_facturacion']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f'{col} → datetime. Nulos: {df[col].isna().sum()}')

# --- Feature: año_atencion ---
df['anio_atencion'] = df['fecha_atencion'].dt.year

# --- Feature: mes_atencion ---
df['mes_atencion'] = df['fecha_atencion'].dt.month

# --- Feature: trimestre ---
df['trimestre'] = df['fecha_atencion'].dt.quarter

# --- Feature: dia_semana ---
df['dia_semana'] = df['fecha_atencion'].dt.dayofweek  # 0=Lunes, 6=Domingo

# --- Feature: hora_registro ---
df['hora_registro'] = df['fecha_registro'].dt.hour

# --- Feature: dias_entre_atencion_y_facturacion ---
df['dias_atencion_facturacion'] = (
    df['fecha_facturacion'] - df['fecha_atencion']
).dt.days

print(f'\n--- Features temporales generadas ---')
temporal_features = ['anio_atencion', 'mes_atencion', 'trimestre', 'dia_semana',
                     'hora_registro', 'dias_atencion_facturacion']
print(df[temporal_features].describe())

# Verificar valores negativos
neg_dias = (df['dias_atencion_facturacion'] < 0).sum()
if neg_dias > 0:
    print(f'\n\u26a0\ufe0f {neg_dias} registros con días negativos (facturación antes de atención)')
else:
    print(f'\n\u2705 Sin valores negativos en dias_atencion_facturacion')

fecha_atencion → datetime. Nulos: 0
fecha_registro → datetime. Nulos: 70
fecha_facturacion → datetime. Nulos: 152

--- Features temporales generadas ---
       anio_atencion  mes_atencion    trimestre   dia_semana  hora_registro  \
count    3126.000000   3126.000000  3126.000000  3126.000000    3056.000000   
mean     2025.326935      5.419386     2.138516     2.989123       9.927029   
std         0.469168      3.261469     1.053303     2.032349       6.072468   
min      2025.000000      1.000000     1.000000     0.000000       0.000000   
25%      2025.000000      3.000000     1.000000     1.000000       5.000000   
50%      2025.000000      5.000000     2.000000     3.000000      10.000000   
75%      2026.000000      8.000000     3.000000     5.000000      15.000000   
max      2026.000000     12.000000     4.000000     6.000000      20.000000   

       dias_atencion_facturacion  
count                2974.000000  
mean                    3.026227  
std                     1.4489

---
## 5. Variables Categóricas (Documentación)

Las siguientes variables categóricas requerirán codificación durante el Modeling.
No se aplica encoding aquí — solo se documenta para la siguiente fase.

| Variable | Valores únicos | Encoding sugerido |
|----------|---------------|------------------|
| sexo | 2 | One-Hot / Binary |
| tipo_atencion | 3 | One-Hot |
| tipo_item | 3 | One-Hot |
| tipo_afiliacion | 2 | Binary |
| eps | ~8 | Label / Target Encoding |
| ciudad | ~5 | Label / Target Encoding |
| sede | ~4 | Label / Target Encoding |
| severidad | 3 | Ordinal (NINGUNA < MEDIA < ALTA) |
| tipo_alerta | 6 | Target multi-clase (no encodear) |

In [11]:
# Documentar cardinalidad de variables categóricas
cat_vars = ['sexo', 'tipo_atencion', 'tipo_item', 'tipo_afiliacion',
            'eps', 'ciudad', 'sede', 'severidad', 'tipo_alerta', 'resultado']

print('=== VARIABLES CATEGÓRICAS ===\n')
for col in cat_vars:
    if col in df.columns:
        nunique = df[col].nunique()
        print(f'  {col}: {nunique} valores únicos → {df[col].unique()[:5]}')

=== VARIABLES CATEGÓRICAS ===

  sexo: 2 valores únicos → ['M' 'F']
  tipo_atencion: 3 valores únicos → ['Urgencias' 'Ambulatoria' 'Hospitalizacion']
  tipo_item: 3 valores únicos → ['consulta' 'examen' 'tratamiento' None]
  tipo_afiliacion: 2 valores únicos → ['Subsidiado' 'Contributivo']
  eps: 6 valores únicos → ['Nueva EPS' 'Sanitas' 'Sura EPS' 'Coosalud' 'Famisanar']
  ciudad: 5 valores únicos → ['Bogota' 'Barranquilla' 'Cali' 'Bucaramanga' 'Medellin']
  sede: 4 valores únicos → ['Sede Urgencias' 'Sede Centro' 'Sede Sur' 'Sede Norte']
  severidad: 3 valores únicos → ['MEDIA' 'ALTA' 'NINGUNA']
  tipo_alerta: 6 valores únicos → ['DIAGNOSTICO_NO_RELACIONADO' 'NO_FACTURADO' 'CONSISTENTE'
 'CANTIDAD_DISCORDANTE' 'SIN_SOPORTE_CLINICO']
  resultado: 2 valores únicos → ['INCONSISTENTE' 'CONSISTENTE']


---
## 6. Variables Textuales (Documentación)

Las siguientes columnas contienen texto libre y son candidatas para la CNN 1D.
No se tokenizan ni se crean embeddings aquí — solo se verifica calidad básica.

In [12]:
text_cols = ['descripcion_diagnostico', 'descripcion', 'descripcion_servicio_facturado']

print('=== VARIABLES TEXTUALES ===\n')
for col in text_cols:
    if col in df.columns:
        non_null = df[col].notna().sum()
        avg_len = df[col].fillna('').str.len().mean()
        max_len = df[col].fillna('').str.len().max()
        unique = df[col].nunique()
        print(f'  {col}:')
        print(f'    No nulos: {non_null} / {len(df)}')
        print(f'    Longitud promedio: {avg_len:.1f} caracteres')
        print(f'    Longitud máxima: {max_len}')
        print(f'    Valores únicos: {unique}')
        print()

=== VARIABLES TEXTUALES ===

  descripcion_diagnostico:
    No nulos: 3126 / 3126
    Longitud promedio: 33.9 caracteres
    Longitud máxima: 55
    Valores únicos: 15

  descripcion:
    No nulos: 3056 / 3126
    Longitud promedio: 25.9 caracteres
    Longitud máxima: 41
    Valores únicos: 18

  descripcion_servicio_facturado:
    No nulos: 2974 / 3126
    Longitud promedio: 25.2 caracteres
    Longitud máxima: 41
    Valores únicos: 18



---
## 7. Validación de Features

In [13]:
new_cols = [c for c in df.columns if c not in original_cols]

print('=== VALIDACIÓN DE FEATURES CREADAS ===\n')
print(f'  Columnas originales: {len(original_cols)}')
print(f'  Nuevas features: {len(new_cols)}')
print(f'  Columnas totales: {df.shape[1]}')

print(f'\n  --- Nuevas features ({len(new_cols)}) ---')
for i, col in enumerate(new_cols, 1):
    dtype = df[col].dtype
    nulls = df[col].isna().sum()
    null_pct = nulls / len(df) * 100
    print(f'    {i:2d}. {col} ({dtype}) — Nulos: {nulls} ({null_pct:.1f}%)')

=== VALIDACIÓN DE FEATURES CREADAS ===

  Columnas originales: 35
  Nuevas features: 20
  Columnas totales: 55

  --- Nuevas features (20) ---
     1. cups_match (int64) — Nulos: 0 (0.0%)
     2. tiene_soporte_clinico (int64) — Nulos: 0 (0.0%)
     3. procedimiento_facturado (int64) — Nulos: 0 (0.0%)
     4. procedimiento_registrado (int64) — Nulos: 0 (0.0%)
     5. diagnostico_disponible (int64) — Nulos: 0 (0.0%)
     6. len_descripcion_diagnostico (int64) — Nulos: 0 (0.0%)
     7. len_descripcion_hc (int64) — Nulos: 0 (0.0%)
     8. len_descripcion_servicio (int64) — Nulos: 0 (0.0%)
     9. diferencia_cantidad (int64) — Nulos: 0 (0.0%)
    10. cantidad_coincide (int64) — Nulos: 0 (0.0%)
    11. valor_unitario_disponible (int64) — Nulos: 0 (0.0%)
    12. valor_total_disponible (int64) — Nulos: 0 (0.0%)
    13. servicio_facturado (int64) — Nulos: 0 (0.0%)
    14. procedimiento_no_facturado (int64) — Nulos: 0 (0.0%)
    15. anio_atencion (int32) — Nulos: 0 (0.0%)
    16. mes_atencion (i

In [14]:
# Muestra de las nuevas features
print('\n  --- Muestra de nuevas features ---')
display(df[new_cols].head(10))


  --- Muestra de nuevas features ---


,cups_match,tiene_soporte_clinico,procedimiento_facturado,procedimiento_registrado,diagnostico_disponible,len_descripcion_diagnostico,len_descripcion_hc,len_descripcion_servicio,diferencia_cantidad,cantidad_coincide,valor_unitario_disponible,valor_total_disponible,servicio_facturado,procedimiento_no_facturado,anio_atencion,mes_atencion,trimestre,dia_semana,hora_registro,dias_atencion_facturacion
0,1,1,1,1,1,44,40,40,0,1,1,1,1,0,2026,1,1,0,6.0,3.0
1,1,1,1,1,1,44,18,18,0,1,1,1,1,0,2026,1,1,0,8.0,3.0
2,0,1,0,1,1,44,22,0,1,0,0,0,0,1,2026,1,1,0,1.0,NaN
3,1,1,1,1,1,34,36,36,0,1,1,1,1,0,2026,4,2,0,11.0,4.0
4,1,1,1,1,1,34,25,25,0,1,1,1,1,0,2026,4,2,0,0.0,4.0
5,1,1,1,1,1,22,36,36,0,1,1,1,1,0,2026,5,2,5,3.0,4.0
6,1,1,1,1,1,22,20,20,0,1,1,1,1,0,2026,5,2,5,4.0,4.0
7,1,1,1,1,1,22,16,16,0,1,1,1,1,0,2026,5,2,5,4.0,4.0
8,1,1,1,1,1,39,40,40,0,1,1,1,1,0,2025,5,2,4,8.0,5.0
9,1,1,1,1,1,39,20,20,0,1,1,1,1,0,2025,5,2,4,4.0,5.0


---
## 8. Exportación del Dataset Enriquecido

In [ ]:
# Guardar dataset con features para uso en Modeling
OUTPUT_PATH = Path('../data/master')

# CSV
csv_out = OUTPUT_PATH / 'master_dataset_features.csv'
df.to_csv(csv_out, index=False)
print(f'\u2705 CSV guardado: {csv_out} ({csv_out.stat().st_size / 1024:.1f} KB)')

# Parquet
parquet_out = OUTPUT_PATH / 'master_dataset_features.parquet'
df.to_parquet(parquet_out, index=False)
print(f'\u2705 Parquet guardado: {parquet_out} ({parquet_out.stat().st_size / 1024:.1f} KB)')

print(f'\nDimensiones finales: {df.shape[0]:,} x {df.shape[1]}')

---
## 9. Resumen Ejecutivo

### Features Clínicas Creadas
| Feature | Propósito | Regla |
|---------|-----------|-------|
| cups_match | ¿CUPS clínico = CUPS facturado? | BR-01 |
| tiene_soporte_clinico | ¿Tiene soporte SI? | BR-02 |
| procedimiento_facturado | ¿Existe prefactura? | BR-01 |
| procedimiento_registrado | ¿Existe registro HC? | BR-02 |
| diagnostico_disponible | ¿Tiene CIE-10? | BR-03 |
| len_descripcion_diagnostico | Longitud texto diagnóstico | CNN 1D |
| len_descripcion_hc | Longitud texto HC | CNN 1D |
| len_descripcion_servicio | Longitud texto servicio | CNN 1D |

### Features de Facturación
| Feature | Propósito | Regla |
|---------|-----------|-------|
| diferencia_cantidad | Cant. realizada - Cant. facturada | BR-06 |
| cantidad_coincide | ¿Cantidades iguales? | BR-06 |
| valor_unitario_disponible | ¿Hay valor unitario? | Dashboard |
| valor_total_disponible | ¿Hay valor total? | Dashboard |
| procedimiento_no_facturado | HC sin PF | BR-01 |

### Features Temporales
| Feature | Propósito |
|---------|----------|
| anio_atencion | Año de la atención |
| mes_atencion | Mes de la atención |
| trimestre | Trimestre |
| dia_semana | Día de la semana (0-6) |
| hora_registro | Hora del registro en HC |
| dias_atencion_facturacion | Días entre atención y facturación |

### Variables Categóricas (para encoding en Modeling)
- sexo, tipo_atencion, tipo_item, tipo_afiliacion, eps, ciudad, sede

### Variables Textuales (para CNN 1D)
- descripcion_diagnostico, descripcion, descripcion_servicio_facturado

### Variables Objetivo
- `resultado` (binaria: CONSISTENTE / INCONSISTENTE)
- `tipo_alerta` (multi-clase: 6 categorías)
- `severidad` (ordinal: NINGUNA, MEDIA, ALTA)

### Recomendaciones para DP-05 / Modeling
1. Aplicar encoding a variables categóricas (One-Hot para baja cardinalidad).
2. Normalizar variables numéricas antes del entrenamiento.
3. Crear embeddings o TF-IDF para variables textuales.
4. Evaluar importancia de features para selección.
5. Aplicar class_weight para manejar desbalance.

### Confirmación
✅ **Dataset enriquecido listo para la fase de Modeling.**  
Total: 20 nuevas features creadas sobre los 35 campos originales.

Master Dataset final:
- 3,126 registros
- 55 columnas